## 📑 Core NPE Potential Inflows Analysis
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AQDLGg)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

import boto3
import os
 
#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
# env = os.environ["S3PREFIX"]

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

som_date = spark.sql(  
f'''
select CAST(PreviousMonthLastWorkingDate AS DATE) ReferenceDateSoM
from `{env_prefix}-cdh`.reference.ref_calendar
where CalendarDate = {run_date!r}
'''
).collect()[0]['ReferenceDateSoM']   
som_date = som_date.strftime('%Y-%m-%d')


reference_date_union = spark.sql(
    f'''
    SELECT MAX(CAST(SnapshotDate AS DATE))  as reference_date_union FROM `{env_prefix}-cdh`.externalfiles.extunionfile
		WHERE CAST(SnapshotDate AS DATE) < {run_date!r} 
    '''
).collect()[0]['reference_date_union']
reference_date_union = reference_date_union.strftime('%Y-%m-%d')

reference_date_maxinflows = spark.sql(
    f'''
    SELECT MAX(BusinessDate)  as reference_date_maxinflows FROM `{env_prefix}-cdh-delivery`.repoint.core_maxinflows
		WHERE BusinessDate < {run_date!r} 
    '''
).collect()[0]['reference_date_maxinflows']
reference_date_maxinflows = reference_date_maxinflows.strftime('%Y-%m-%d')

##### Schema of the table

In [0]:
table_name = 'Core_NPEPotentialInflowsAnalysis'

schema = (
"BusinessDate date,"
"CDI string,"
"AgreementKey string,"
"ExposureNumber string,"
"Balance decimal(19,2),"
"MaxInflowFlag string,"
"InflowMonth string,"
"MaxInflowReason string,"
"TargetDate date,"
"TargetWeek bigint,"
"InflowStatus string,"
"SavedReasonGroup string,"
"ContactCategory string,"
"ContactCategoryGroup string,"
"AssignmentChannel string,"
"Actions string,"
"Pipeline string,"
"ServicedFlag string,"
"SavedDate date,"
"DOD_DPDs int,"
"DOD_Bucket int,"
"DOD_MAX_Bucket int,"
"NPEStatusCFPLNextMonth string,"
"NPEInflowMonthCFPLNextMonth string,"
"InsertTimeStamp timestamp"
)

In [0]:
%python
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"
delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date,  # type: ignore
)


##### Union File

In [0]:
UnionFile = spark.sql(
f'''
select * 
from `{env_prefix}-cdh`.externalfiles.extunionfile 
where SnapshotDate = {reference_date_union!r} 
and IngestionTimeStamp = (Select Max(IngestionTimeStamp) from `{env_prefix}-cdh`.externalfiles.extunionfile where SnapshotDate = '{reference_date_union}')
''')   
UnionFile.createOrReplaceTempView("UnionFile")

##### Modifications

In [0]:
Modifications = spark.sql(
f'''
select DISTINCT 
	M.OldExposureKey
 ,	M.Category
 ,	m.ModificationDate
 ,	RANK() OVER (PARTITION BY M.OldExposureKey ORDER BY m.ModificationDate ASC, M.Category) AS RNKMods
from `{env_prefix}-cdh-delivery`.repoint.core_modifications M 
where BusinessDate = {run_date!r}
'''
)   
Modifications.createOrReplaceTempView("Modifications")

##### NPEPlanningData

In [0]:
NPEPlanningData = spark.sql(
f'''
SELECT  ExposureKey
      , Status
      , ActiveRequestMIS
      , CASE WHEN ActiveRequestMIS = 'Yes' AND Status = 'Ενεργή'   THEN 'Ενεργή'
             WHEN ActiveRequestMIS = 'Yes' AND Status = 'Ολοκληρωμένη' THEN 'Ολοκληρωμένη'
             ELSE NULL END AS Pipeline        
FROM `{env_prefix}-cdh-delivery`.repoint.core_NPEPlanningdata
WHERE BusinessDate = {run_date!r}

''')   
NPEPlanningData.createOrReplaceTempView("NPEPlanningData")

##### WeekOfMonth

In [0]:
WeekOfMonth = spark.sql(
f'''
SELECT 
               CASE WHEN Fd.DayName = 'Sunday' and c.DayName <> 'Sunday'  THEN WeekOfMonth + 1
        	            WHEN Fd.DayName = 'Sunday' and c.DayName = 'Sunday'   THEN WeekOfMonth 
        	            WHEN Fd.DayName <> 'Sunday' and c.DayName <> 'Sunday' THEN WeekOfMonth 
        	            WHEN Fd.DayName <> 'Sunday' and c.DayName = 'Sunday'  THEN WeekOfMonth -1
        	            END AS WeekOfMonthNew
             , C.*
from `{env_prefix}-cdh`.reference.ref_calendar  C
LEFT JOIN (SELECT CalendarDate AS FirstDayMonth , DayName as DayName
           FROM `{env_prefix}-cdh`.reference.ref_calendar WHERE CalendarDate = FirstDateOfMonth)  FD
ON C.FirstDateOfMonth = fd.FirstDayMonth
''')   
WeekOfMonth.createOrReplaceTempView("WeekOfMonth")

##### MonthBusinessDates

In [0]:
MonthBusinessDates = spark.sql(
f'''
SELECT DISTINCT CalendarDate AS REF_DATE
FROM `{env_prefix}-cdh`.reference.ref_calendar 
WHERE 1=1
AND CalendarDate > {som_date!r}		 
AND CalendarDate <= {run_date!r}
''')   
MonthBusinessDates.createOrReplaceTempView("MonthBusinessDates")

##### RefDates

In [0]:
RefDates = spark.sql(
f'''
SELECT dp.BusinessDate as RefDate, MAX(dp.IngestionTimeStamp) IngestionTimeStamp
FROM `{env_prefix}-cdh`.externalfiles.extdod_dpd_fcy    AS DP
INNER JOIN MonthBusinessDates AS BD ON DP.BusinessDate= BD.REF_DATE
GROUP BY dp.BusinessDate
'''
)   
RefDates.createOrReplaceTempView("RefDates")

##### CFPLNextMonth_1

In [0]:
CFPLNextMonth_1 = spark.sql(
f'''
WITH extdod_dpd_fcy AS (
    SELECT BusinessDate,
           AC_SRC_STM_CODE, 
           AC_CODE,
            COALESCE(
                TRY_TO_DATE(
                    REGEXP_REPLACE(TRIM(DOD_DPD_CNTER_STRT_DT), '\\s+', ''),
                    'ddMMMyyyy'
                ),
                TRY_TO_DATE(
                    REGEXP_REPLACE(TRIM(DOD_DPD_CNTER_STRT_DT), '\\s+', ''),
                    'dMMMyyyy'
                ),
                TRY_TO_DATE(
                    TRIM(DOD_DPD_CNTER_STRT_DT),
                    'd/M/yyyy'
                )
            ) AS DOD_DPD_CNTER_STRT_DT,
            COALESCE(
                TRY_TO_DATE(
                    REGEXP_REPLACE(TRIM(DOD_DPD_CNTER_END_DT), '\\s+', ''),
                    'ddMMMyyyy'
                ),
                TRY_TO_DATE(
                    REGEXP_REPLACE(TRIM(DOD_DPD_CNTER_END_DT), '\\s+', ''),
                    'dMMMyyyy'
                ),
                TRY_TO_DATE(
                    TRIM(DOD_DPD_CNTER_END_DT),
                    'd/M/yyyy'
                )
            ) AS DOD_DPD_CNTER_END_DT,
            IngestionTimeStamp
    FROM `{env_prefix}-cdh`.externalfiles.extdod_dpd_fcy 
)

select DP.BusinessDate
, DP.AC_SRC_STM_CODE  
, DP.AC_CODE  
, TO_DATE(DP.DOD_DPD_CNTER_STRT_DT, 'd/M/yyyy') AS DOD_DPD_CNTER_STRT_DT
, TO_DATE(DP.DOD_DPD_CNTER_END_DT, 'd/M/yyyy')  AS DOD_DPD_CNTER_END_DT  
, RANK() OVER (PARTITION BY DP.AC_CODE ORDER BY TO_DATE(DP.DOD_DPD_CNTER_END_DT, 'd/M/yyyy') DESC) AS RNKMods 
, RANK() OVER (PARTITION BY DP.AC_CODE ORDER BY DP.BusinessDate DESC)        AS RNKReferenceDate
FROM extdod_dpd_fcy   AS DP
inner join RefDates  RD on DP.BusinessDate=RD.RefDate AND DP.IngestionTimeStamp = RD.IngestionTimeStamp
INNER JOIN UnionFile    P ON P.ExposureKey = DP.AC_CODE
WHERE EBAStatus = 'FPL' AND FPLUnderProbationFlag = 1
''')   
CFPLNextMonth_1.createOrReplaceTempView("CFPLNextMonth_1")

##### CFPLNextMonth_2

In [0]:
CFPLNextMonth_2 = spark.sql(
f'''
	SELECT BusinessDate
           ,AC_SRC_STM_CODE
           ,AC_CODE
           ,DOD_DPD_CNTER_STRT_DT
           ,DOD_DPD_CNTER_END_DT		
FROM CFPLNextMonth_1 
WHERE RNKReferenceDate = 1 
AND DOD_DPD_CNTER_STRT_DT <= {run_date!r} 
AND  DOD_DPD_CNTER_STRT_DT > last_day(add_months({run_date!r},-1))
AND ((DOD_DPD_CNTER_END_DT <= {run_date!r} AND  DOD_DPD_CNTER_END_DT > last_day(add_months({run_date!r},-1)) ) or DOD_DPD_CNTER_END_DT = '9999-12-31')
''')   
CFPLNextMonth_2.createOrReplaceTempView("CFPLNextMonth_2")

##### DoublePotentialInflow

In [0]:
DoublePotentialInflow_Step0 = spark.sql(
f'''
SELECT c1.*
from CFPLNextMonth_1 c1
INNER JOIN CFPLNextMonth_2  c2
ON c1.AC_CODE = c2.AC_CODE
''')   
DoublePotentialInflow_Step0.createOrReplaceTempView("DoublePotentialInflow_Step0")

In [0]:
DoublePotentialInflow = spark.sql(
f'''
SELECT DISTINCT AC_CODE
	
	FROM DoublePotentialInflow_Step0       
	GROUP BY AC_CODE
	HAVING MAX(DOD_DPD_CNTER_END_DT) = '9999-12-31'
''')   
DoublePotentialInflow.createOrReplaceTempView("DoublePotentialInflow")

In [0]:
DoublePotentialInflow_1 = spark.sql(
f'''
SELECT DISTINCT AC_CODE
FROM DoublePotentialInflow_Step0      
GROUP BY AC_CODE
HAVING MAX(DOD_DPD_CNTER_STRT_DT) = MIN(DOD_DPD_CNTER_STRT_DT)
''')   
DoublePotentialInflow_1.createOrReplaceTempView("DoublePotentialInflow_1")

In [0]:
DoublePotentialInflow_final = spark.sql(
f'''
SELECT DISTINCT s.AC_CODE
		             , 'Pending' AS InflowStatus
		FROM DoublePotentialInflow_Step0  s
		INNER JOIN  DoublePotentialInflow d
		on d.AC_CODE = s.AC_CODE
		LEFT JOIN DoublePotentialInflow_1 f
		ON F.AC_CODE = S.AC_CODE
		WHERE RNKReferenceDate = 1 AND DOD_DPD_CNTER_END_DT = '9999-12-31' AND F.AC_CODE IS NULL
  
  UNION

  SELECT DISTINCT s.AC_CODE
		              , 'Saved' AS InflowStatus
		FROM DoublePotentialInflow_Step0  s
		INNER JOIN  DoublePotentialInflow d
		on d.AC_CODE = s.AC_CODE
		LEFT JOIN DoublePotentialInflow_1 f
		ON F.AC_CODE = S.AC_CODE
		WHERE RNKReferenceDate = 1 AND (DOD_DPD_CNTER_END_DT <= {run_date!r} AND  DOD_DPD_CNTER_END_DT > last_day(add_months({run_date!r}, -1)))  AND F.AC_CODE IS NULL 
''')   
DoublePotentialInflow_final.createOrReplaceTempView("DoublePotentialInflow_final")

##### CFPLNextMonth

In [0]:
CFPLNextMonth = spark.sql(
f'''
SELECT DISTINCT u.*
		     , AC_CODE
			 , AC_SRC_STM_CODE
			 , DOD_DPD_CNTER_END_DT
			 , DOD_DPD_CNTER_STRT_DT
			 , CASE WHEN AC_SRC_STM_CODE = 'HST' OR ExposureSourceSystemKey = 'ATO' OR ProductCode IN ('6090','6091') OR ProductType = 500 OR
        	                                        (ExposureSegmentation='6' AND ProductType IN ('180', '213', '293', '493', '570', '50038', '82', '83')) THEN 'No'
                     ELSE 'Yes' END AS ServicedFlag
	
		FROM UnionFile              AS  u                   
		LEFT JOIN RefDates		 AS RD
		LEFT JOIN CFPLNextMonth_2   AS  f
		ON U.ExposureKey = f.AC_CODE AND f.BusinessDate = rd.RefDate 
		WHERE ExposureSegmentation > 3 AND cast(replace(IFRSGrossCarryingAmount,",",'') as decimal(19,2)) > 0 AND (HFSFlag = 0 OR HFSFlag is null)    
        AND u.BucketCategoryWithMateriality < 4	

'''
)   
CFPLNextMonth.createOrReplaceTempView("CFPLNextMonth")

##### Perimeter

In [0]:
Perimeter = spark.sql(
f'''
SELECT	A.*
			,	RANK() OVER (PARTITION BY A.AccountNumber ORDER BY InputFile ASC, Asset) AS RNK
		
		FROM
		(
		SELECT 
				m.ExposureNumber							AS AccountNumber							
      		,   m.Asset
			,   m.Balance
			,   m.TargetDate
			,   m.MaxInflowFlag
			,   m.InflowMonth
			,   m.InflowMonthCode
			,   m.DODStartDate
			,   m.DOD_DPDs
			,   m.DOD_Bucket
			,   m.DOD_MAX_Bucket
			,   m.ServicedFlag
			,   m.HFS
			,	1										AS InputFile
		FROM `{env_prefix}-cdh-delivery`.repoint.core_maxinflows m       
		WHERE 1=1
			  AND m.BusinessDate = {reference_date_maxinflows!r}	
		
		UNION 
		 
		SELECT	u.ExposureNumber                                  AS AccountNumber
		    ,   c.ECAAssetClass                              AS Asset
			,   cast(replace(c.IFRSGrossCarryingAmount,",",'') as decimal(19,2))   AS Balance
			,   DATEADD(DAY,30,DOD_DPD_CNTER_STRT_DT)         AS TargetDate
			,   'CFPL Max Inflow'                               AS MaxInflowFlag
			,   date_format(DATEADD(DAY, 1, {run_date!r}), 'yyyy-MM') AS InflowMonth
			,   'M1'                                            AS InflowMonthCode
			,   DOD_DPD_CNTER_STRT_DT                         AS DODStartDate
			,   0                                               AS DOD_DPDs
			,   0                                               AS DOD_Bucket
			,   2                                               AS DOD_MAX_Bucket
			,   ServicedFlag
			,   c.HFSFlag as HFS
			,	2	                             AS InputFile
		FROM CFPLNextMonth  c
		LEFT JOIN UnionFile u
		ON c.AC_CODE = u.ExposureKey
		)		as a


'''
)   
Perimeter.createOrReplaceTempView("Perimeter")

##### Final_Step1

In [0]:
Final_Step1 = spark.sql(
f'''
SELECT DISTINCT 
		        u.ECAAssetClass
              , u.ExposureNumber
			  , u.ExposureKey
        	  , u.BorrowerKey
        	  , u.AgreementKey
              , cast(replace(u.IFRSGrossCarryingAmount,",",'') as decimal(19,2)) as IFRSGrossCarryingAmount
              , u.ExposureSourceSystemKey
              , u.ExposureSegmentation
              , u.EBAStatus
              , u.FPLUnderProbationFlag
              , u.NonCoreFlag
              , u.ProductCode
		      , d.Asset
              , d.AccountNumber
              , d.DODStartDate
              , d.Balance
              , d.MaxInflowFlag
              , d.DOD_DPDs
              , d.DOD_Bucket
              , d.DOD_MAX_Bucket
              , d.TargetDate
              , d.InflowMonth
              , d.InflowMonthCode
              , d.ServicedFlag
              , d.HFS	  
        	  , u.BucketCategoryWithMateriality
			  , m.Category
			  , u.SnapshotDate as Snapshot_Date
        FROM UnionFile                        u 
		INNER JOIN Perimeter                d
        ON d.AccountNumber = u.ExposureNumber   AND d.RNK = 1
		LEFT JOIN Modifications               m
        ON m.OldExposureKey = u.ExposureNumber AND m.RNKMods = 1
		WHERE u.ExposureSourceSystemKey IN ('CMS', 'HSL', 'HST', 'LMS', 'SGL', 'SSE') AND u.EBAStatus IN ('PL','FPL') AND
		      u.ExposureSegmentation > 3 AND cast(replace(u.IFRSGrossCarryingAmount,",",'') as decimal(19,2)) >0  AND (u.HFSFlag = 0 OR u.HFSFlag is null)

'''
)   
Final_Step1.createOrReplaceTempView("Final_Step1")

##### DOD_DPD_FCY

In [0]:
DOD_DPD_FCY = spark.sql(
f'''
WITH extdod_dpd_fcy AS (
SELECT BusinessDate,
       AC_SRC_STM_CODE,
       AC_CODE,
       COALESCE(
                TRY_TO_DATE(
                    REGEXP_REPLACE(TRIM(DOD_DPD_CNTER_STRT_DT), '\\s+', ''),
                    'ddMMMyyyy'
                ),
                TRY_TO_DATE(
                    REGEXP_REPLACE(TRIM(DOD_DPD_CNTER_STRT_DT), '\\s+', ''),
                    'dMMMyyyy'
                ),
                TRY_TO_DATE(
                    TRIM(DOD_DPD_CNTER_STRT_DT),
                    'd/M/yyyy'
                )
            ) AS DOD_DPD_CNTER_STRT_DT,
            COALESCE(
                TRY_TO_DATE(
                    REGEXP_REPLACE(TRIM(DOD_DPD_CNTER_END_DT), '\\s+', ''),
                    'ddMMMyyyy'
                ),
                TRY_TO_DATE(
                    REGEXP_REPLACE(TRIM(DOD_DPD_CNTER_END_DT), '\\s+', ''),
                    'dMMMyyyy'
                ),
                TRY_TO_DATE(
                    TRIM(DOD_DPD_CNTER_END_DT),
                    'd/M/yyyy'
                )
            ) AS DOD_DPD_CNTER_END_DT,
       IngestionTimeStamp
FROM `{env_prefix}-cdh`.externalfiles.extdod_dpd_fcy 
)

SELECT	DP.BusinessDate
        ,DP.AC_SRC_STM_CODE
        ,DP.AC_CODE
        ,TO_DATE(DP.DOD_DPD_CNTER_STRT_DT, 'd/M/yyyy') AS DOD_DPD_CNTER_STRT_DT
    	,TO_DATE(DP.DOD_DPD_CNTER_END_DT, 'd/M/yyyy')  AS DOD_DPD_CNTER_END_DT
		,RANK() OVER (PARTITION BY DP.AC_CODE ORDER BY TO_DATE(DP.DOD_DPD_CNTER_END_DT, 'd/M/yyyy') ASC) AS RNKMods  
		,RANK() OVER (PARTITION BY DP.AC_CODE ORDER BY DP.BusinessDate ASC)        AS RNKRefsDate
		
		FROM extdod_dpd_fcy  AS DP
		INNER JOIN RefDates      AS RD ON RD.RefDate = dp.BusinessDate AND RD.IngestionTimeStamp = dp.IngestionTimeStamp
		INNER JOIN Final_Step1   AS  P ON P.ExposureKey = DP.AC_CODE
WHERE (TO_DATE(DP.DOD_DPD_CNTER_END_DT, 'd/M/yyyy') <= {run_date!r} AND TO_DATE(DP.DOD_DPD_CNTER_END_DT, 'd/M/yyyy') > last_day(add_months({run_date!r},-1)) )
		   OR TO_DATE(DP.DOD_DPD_CNTER_END_DT, 'd/M/yyyy') = '9999-12-31'
'''
)   
DOD_DPD_FCY.createOrReplaceTempView("DOD_DPD_FCY")

##### DOD_DPD_FCY_MaxEndDate

In [0]:
DOD_DPD_FCY_MaxEndDate = spark.sql(
f'''
SELECT AC_CODE
		
		FROM DOD_DPD_FCY
		WHERE RNKRefsDate = 1
		GROUP BY AC_CODE
		HAVING MAX(DOD_DPD_CNTER_END_DT) <> '9999-12-31'
'''
)   
DOD_DPD_FCY_MaxEndDate.createOrReplaceTempView("DOD_DPD_FCY_MaxEndDate")


##### Final_Step2

In [0]:
Final_Step2 = spark.sql(
f'''
SELECT DISTINCT 
 	        d.ECAAssetClass
              , d.ExposureNumber
	      , d.ExposureKey
              , d.BorrowerKey
              , d.AgreementKey
              , d.IFRSGrossCarryingAmount
              , d.ExposureSourceSystemKey
              , d.ExposureSegmentation
              , d.EBAStatus
              , d.FPLUnderProbationFlag
              , d.NonCoreFlag
              , d.ProductCode        	      
              , CASE WHEN f.DOD_DPD_CNTER_END_DT <= {run_date!r} AND f.DOD_DPD_CNTER_END_DT > last_day(add_months({run_date!r},-1) )
			          AND f.DOD_DPD_CNTER_END_DT <= d.TargetDate THEN 'Saved'   
        	         WHEN {run_date!r} < TargetDate THEN 'Pending'
        			 ELSE 'Realized inflow' 
					 END AS InflowStatus
			  , CASE WHEN f.DOD_DPD_CNTER_END_DT <= {run_date!r} AND f.DOD_DPD_CNTER_END_DT >  last_day(add_months({run_date!r},-1) )
			          AND f.DOD_DPD_CNTER_END_DT <= d.TargetDate THEN f.DOD_DPD_CNTER_END_DT 
			         END AS SavedDate
			  , CASE WHEN f.DOD_DPD_CNTER_END_DT <= {run_date!r} AND f.DOD_DPD_CNTER_END_DT >  last_day(add_months({run_date!r},-1) )
			          AND f.DOD_DPD_CNTER_END_DT <= d.TargetDate THEN 1
			         ELSE 0 
					 END AS SavedDateRank
              , d.Asset
              , d.AccountNumber
              , d.DODStartDate
              , d.Balance
              , d.MaxInflowFlag
              , d.DOD_DPDs
              , d.DOD_Bucket
              , d.DOD_MAX_Bucket
              , d.TargetDate
              , d.InflowMonth
              , d.InflowMonthCode
              , d.ServicedFlag
              , d.HFS
        	  , d.BucketCategoryWithMateriality
			  , d.Category
        
		FROM Final_Step1       AS  D
		Left Join RefDates	AS RD
		LEFT JOIN DOD_DPD_FCY  AS  F
		ON d.ExposureKey = f.AC_CODE AND f.BusinessDate = rd.RefDate AND f.RNKMods = 1

'''
)   
Final_Step2.createOrReplaceTempView("Final_Step2")


##### Final_Step3

In [0]:
Final_Step3 = spark.sql(
f'''
SELECT DISTINCT 
		        F.*
              , CASE WHEN InflowStatus = 'Saved' AND Category = 'Extension'                                                     THEN 'PJ extension'
        	             WHEN InflowStatus = 'Saved' AND (Category LIKE '%A4%' OR Category LIKE '%A5%' OR Category LIKE '%A9%') THEN 'A4 A5 A9'   
        			     WHEN InflowStatus = 'Saved' 
				  	  AND (Category NOT LIKE '%A4%' OR Category NOT LIKE '%A5%' OR Category NOT LIKE '%A9%') 
				  	  AND Category IS NOT NULL                                                                                    THEN 'Other Modifications'  
        			     WHEN InflowStatus = 'Saved'                                                                                  THEN 'Collection'
        			     WHEN InflowStatus <> 'Saved'                                                                                 THEN NULL 
					 END AS SavedReasonGroup
			  ,	RANK() OVER (PARTITION BY F.ExposureNumber ORDER BY SavedDateRank DESC) AS RNKMods
		FROM Final_Step2  AS F


'''
)   
Final_Step3.createOrReplaceTempView("Final_Step3")


##### AssignmentMapping

In [0]:
AssignmentMapping = spark.sql(
f'''
SELECT  
		    ExposureKey 
        	  , CASE WHEN AssignmentChannel IN ('Legal Agencies', 'Not Boarded', 'Outsourcing agencies', 'unhandled', 'DNC_IRT') THEN AssignmentChannel  
        	         WHEN AssignmentChannel = 'DNC_Collection' THEN CASE WHEN CourseOfAction = 'Double Treatment' THEN 'Double Treatment CC-IRT'
        			                                                       ELSE 'DNC_Collection' END
                 WHEN AssignmentChannel = 'HUB RMs' THEN CASE WHEN CourseOfAction = 'Double Treatment' AND AssignmentSegmentDetailed in ('HUBS PE', 'LowInstallment') THEN 'Hub RMs' 
        			                                                WHEN CourseOfAction = 'Double Treatment' THEN 'Double Treatment CC- Hubs'
        															ELSE 'Hub RMs' END
                     END AS AssignmentChannel
              , CASE WHEN CourseOfAction in ('collection', 'Collection')                         THEN 'Collection'
        	             WHEN CourseOfAction IN ('N.3869','Not Boarded','Unhandled','unhandled')                               THEN 'No Action'
        			     WHEN AssignmentSegmentDetailed IN ('LowInstallment','PJ_Extension','PJ_Extension_StepUp') THEN 'Extension'
        			     ELSE 'Modification Normal' END AS Actions
       
        FROM `{env_prefix}-cdh-delivery`.repoint.core_assignmentdatamapping
		WHERE 1 = 1
		  AND BusinessDate = {run_date!r}  
''')   
AssignmentMapping.createOrReplaceTempView("AssignmentMapping")

##### Strongest Action Step 1

In [0]:
interaction = spark.sql(
f'''
select  distinct ExposureKey,InteractionDate, InteractionTypeKey, 
  CASE WHEN m.Hierarchy in (4,5,6) THEN 4 ELSE m.Hierarchy END AS HierarchyNew, m.Hierarchy, Activity_Name
from `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) as ia
inner join `{env_prefix}-reportinputfiles`.repoint.core_qcabactivitiesmapping m on cast(ia.InteractionTypeKey as string) = cast(m.Activity_Code as string)
where 1=1  -- ia.BusinessDate =  {run_date!r}
and ia.InteractionDate > last_day(add_months({run_date!r},-1))
and ia.InteractionDate <= {run_date!r}
and m.Base_Category not in('Legal Action', 'no attempt')
and m.Activity_Group in ('Modifications', 'Collections','Collections/Modifications')
and ia.PortfolioName = 'Core Retail'
and ia.InteractionTypeKey <> '100'
''')
interaction.createOrReplaceTempView("interaction")

In [0]:
StrongestAction_Step1 = spark.sql(
f'''
select  distinct p.AccountNumber
, int.Hierarchy
, int.HierarchyNew
, int.InteractionDate
, int.InteractionTypeKey
, CASE WHEN int.Hierarchy='2'                                                                  THEN 'Contact'
        	               WHEN int.Hierarchy='3'                                                                  THEN 'Rest RPC Communications'
        			       WHEN int.Hierarchy='4' AND int.Activity_Name = '621_IRT_PJ_EXT_ΑΡΝΗΤΙΚΟΣ ΓΙΑ ΠΑΡΑΤΑΣΗ'  THEN 'Negative for Extension'
        			       WHEN int.Hierarchy='4' AND int.Activity_Name <> '621_IRT_PJ_EXT_ΑΡΝΗΤΙΚΟΣ ΓΙΑ ΠΑΡΑΤΑΣΗ' THEN 'Refusal'
        			       WHEN int.Hierarchy='5'                                                                  THEN 'Promise for payment'
        			       WHEN int.Hierarchy='6' AND int.Activity_Name = '620_IRT_PJ_EXT_ΘΕΤΙΚΟΣ ΓΙΑ ΠΑΡΑΤΑΣΗ'    THEN 'Positive for Extension'
        			       WHEN int.Hierarchy='6' AND int.Activity_Name <> '620_IRT_PJ_EXT_ΘΕΤΙΚΟΣ ΓΙΑ ΠΑΡΑΤΑΣΗ'   THEN 'Positive'
        			       ELSE 'Not Found' END AS ContactCategory
, RANK() OVER (PARTITION BY p.AccountNumber ORDER BY int.HierarchyNew DESC, InteractionDate DESC, InteractionTypeKey DESC) AS RNK

from Perimeter p  --AssignmentMapping p
inner join interaction int
on p.AccountNumber = int.ExposureKey

'''
)
StrongestAction_Step1.createOrReplaceTempView("StrongestAction_Step1")


##### StrongestAction

In [0]:
StrongAction = spark.sql(
f'''

        SELECT  *
        	      , CASE WHEN ContactCategory IN ('Positive', 'Positive for Extension') THEN 'Positive'
        	             WHEN ContactCategory IN ('Refusal','Negative for Extension')   THEN 'Negative'
        			     ELSE ContactCategory END AS ContactCategoryGroup
       
        FROM StrongestAction_Step1
		WHERE RNK = 1

'''
)
StrongAction.createOrReplaceTempView("StrongAction")


##### Gefyra

In [0]:
Gefyra = spark.sql(
f'''
        SELECT DISTINCT Account_Number FROM (
            SELECT * FROM `{env_prefix}-cdh`.reference.extfile_implementations_gefyra_i_31_01_2023
            UNION ALL
            SELECT * FROM `{env_prefix}-cdh`.reference.extfile_implementations_gefyra_ii_31_01_2023
        )
 '''
)
Gefyra.createOrReplaceTempView("Gefyra")

In [0]:
extpaymentjumptotalperimeter = spark.sql(
f'''
   select p.*
   from `{env_prefix}-cdh`.externalfiles.extpaymentjumptotalperimeter p
   inner join (select max(IngestionTimeStamp) IngestionTimestamp, BusinessDate from `{env_prefix}-cdh`.externalfiles.extpaymentjumptotalperimeter group by BusinessDate) b
   on p.BusinessDate = b.BusinessDate and p.IngestionTimeStamp = b.IngestionTimestamp
   where p.BusinessDate = (select max(BusinessDate) from `{env_prefix}-cdh`.externalfiles.extpaymentjumptotalperimeter where BusinessDate <= {run_date!r})
'''
)
extpaymentjumptotalperimeter.createOrReplaceTempView("extpaymentjumptotalperimeter")

# Check if the dataset is empty and print a message accordingly
if extpaymentjumptotalperimeter.count() == 0:
    print(f"The required external file extpaymentjumptotalperimeter dataset for dates <= {run_date!r} is empty")

##### NPEPotentialInflowsAnalysis_v1

In [0]:
NPEPotentialInflowsAnalysis_v1 = spark.sql(
f'''
   select distinct f.*
   ,w.WeekOfMonthNew   AS TargetWeek
   , CASE WHEN p.ExposureKey = f.ExposureNumber     THEN 'PaymentJump'
        	            WHEN g.Account_Number = f.ExposureNumber THEN 'Gefyra'
        			    ELSE 'Due to bucket' end  as MaxInflowReason 
   , IFNULL(ContactCategory, 'Not found') ContactCategory
   , IFNULL(ContactCategoryGroup, 'Not found') ContactCategoryGroup
   , Pipeline
   , AssignmentChannel
   , Actions
 from Final_Step3  as f 
 left join Gefyra  as g 
 on g.Account_Number = f.ExposureNumber 
 left join extpaymentjumptotalperimeter p
 on p.ExposureKey = f.ExposureNumber
 LEFT JOIN StrongAction                        s
ON s.AccountNumber = f.ExposureNumber	
LEFT JOIN AssignmentMapping                   m
ON m.ExposureKey = f.ExposureNumber			
LEFT JOIN NPEPlanningData                     n
ON n.ExposureKey = f.ExposureNumber	 			
LEFT JOIN WeekOfMonth                         w
ON w.CalendarDate = f.TargetDate
WHERE RNKMods = 1
  '''
)
NPEPotentialInflowsAnalysis_v1.createOrReplaceTempView("NPEPotentialInflowsAnalysis_v1")

##### NPEPotentialInflowsAnalysis

In [0]:
NPEPotentialInflowsAnalysis = spark.sql(
f'''
 SELECT f.* 
		     , CASE WHEN InflowMonth = date_format({run_date!r},'yyyy-MM') AND d.AC_CODE = f.ExposureKey 
			       AND f.InflowStatus = 'Saved' THEN d.InflowStatus END AS NPEStatusCFPLNextMonth
			 , CASE WHEN InflowMonth = date_format({run_date!r},'yyyy-MM') AND d.AC_CODE = f.ExposureKey 
			       AND f.InflowStatus = 'Saved' THEN date_format(add_months({run_date!r}, 1),'yyyy-MM') END AS NPEInflowMonthCFPLNextMonth
		FROM NPEPotentialInflowsAnalysis_v1                            f
        LEFT JOIN DoublePotentialInflow_final d
		ON d.AC_CODE = f.ExposureKey

  '''
)
NPEPotentialInflowsAnalysis.createOrReplaceTempView("NPEPotentialInflowsAnalysis")

##### final_df

In [0]:
final_df = spark.sql(
f'''

SELECT 
					cast({run_date!r} as date) AS BusinessDate      
	    	,       cast(BorrowerKey as string) AS CDI
			,		cast(AgreementKey as string) AS AgreementKey
			,       cast(ExposureNumber as string) AS ExposureNumber
			,       cast(IFRSGrossCarryingAmount as decimal(19,2)) AS Balance
			,       cast(MaxInflowFlag as string) AS MaxInflowFlag
			,       cast(InflowMonth as string) AS InflowMonth
			,       cast(MaxInflowReason as string) AS MaxInflowReason
			,       cast(TargetDate as date) AS TargetDate
			,       cast(TargetWeek as bigint) AS TargetWeek
			,       cast(InflowStatus as string) AS InflowStatus
			,       cast(SavedReasonGroup as string) AS SavedReasonGroup
			,       cast(ContactCategory as string) AS ContactCategory        
			,       cast(ContactCategoryGroup as string) AS ContactCategoryGroup	 
			,       cast(AssignmentChannel as string) AS AssignmentChannel
			,       cast(Actions as string) AS Actions
			,       cast(Pipeline as string) AS Pipeline
			,       cast(ServicedFlag as string) AS ServicedFlag
			,       cast(SavedDate as date) AS SavedDate
			,       cast(DOD_DPDs as int) AS DOD_DPDs
			,       cast(DOD_Bucket as int) AS DOD_Bucket
			,       cast(DOD_MAX_Bucket as int) AS DOD_MAX_Bucket
			,       cast(NPEStatusCFPLNextMonth as string) AS NPEStatusCFPLNextMonth
			,       cast(NPEInflowMonthCFPLNextMonth as string) AS NPEInflowMonthCFPLNextMonth
			,		cast(getdate() as timestamp) AS InsertTimeStamp
		FROM NPEPotentialInflowsAnalysis
'''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_NPEPotentialInflowsAnalysis: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureNumber"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
npe_potential_inflows = spark.sql(
    f'''
    select * from {delivery_table_path}
    where BusinessDate = '{run_date}'
    '''
)
npe_potential_inflows.createOrReplaceTempView("npe_potential_inflows")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/NPE_Potential_Inflows/"
file_name = "NPE_Potential_Inflows_" +run_date+ ".csv"
npe_potential_inflows.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)